# 03. Preparación de características y etiqueta para regresión

Se parte de la **base común 2023–2025**, sin modificarla ni repetir su limpieza. Cada fila representa un egreso hospitalario. Se preparan ocho características (`X`) y los días reales de estadía (`y`) para entrenar modelos en otra fase; aquí no se entrena ni se evalúa una regresión.

**Alcance retrospectivo:** `DIAG1_CATEGORIA` deriva del diagnóstico principal de egreso. Utilizarlo no demuestra capacidad de predecir la duración al ingresar al hospital ni permite atribuir relaciones causales.

Ejecutar desde la raíz del proyecto con su entorno `.venv` y las dependencias de `requirements.txt`.

## 1. Características, etiqueta y trazabilidad

| Característica | Justificación de la selección inicial |
| --- | --- |
| `PERTENENCIA_ESTABLECIMIENTO_SALU` | Distinguir la pertenencia al SNSS, sin equipararla automáticamente con público/privado. |
| `SEXO` | Representar el sexo registrado, conservando todas sus categorías. |
| `GRUPO_EDAD` | Incorporar el intervalo de edad sin inventar una edad exacta o un punto medio. |
| `ETNIA` | Representar la identificación con alguna etnia; no equivale a una etnia específica. |
| `GLOSA_PAIS_ORIGEN` | Distinguir las categorías de origen disponibles, sin atribuir un país detallado. |
| `PREVISION` | Incorporar la previsión de salud al ingreso como categoría, no como cantidad. |
| `REGION_RESIDENCIA` | Incorporar residencia con menos fragmentación que la comuna. |
| `DIAG1_CATEGORIA` | Representar el diagnóstico con menos detalle que el código completo y más que el capítulo. |

La etiqueta es **`DIAS_ESTADA`**, numérica y en días. `ANO_FUENTE` e `ID_REGISTRO` se conservan aparte como identificador compuesto de la fila; no identifican a una persona.

Quedan fuera de `X`: `CONDICION_EGRESO`, `ERROR`, los identificadores, `ANO_EGRESO`, las marcas de calidad y los estados de correspondencia. También se excluyen la comuna, las glosas redundantes, las otras representaciones de `DIAG1` y `DIAG2` (causa externa, no segunda enfermedad). **`ESTADIA_MAYOR_365` se excluye porque deriva directamente de la etiqueta.** Las 36 columnas permanecen intactas en la base común.

In [2]:
from pathlib import Path
from importlib.metadata import version
import hashlib

import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from sklearn.preprocessing import OneHotEncoder

ARCHIVO_COMUN = Path("datos/procesados/egresos_2023_2025_base_comun.parquet")
SALIDA = Path("datos/procesados/regresion")
CARACTERISTICAS = [
    "PERTENENCIA_ESTABLECIMIENTO_SALU", "SEXO", "GRUPO_EDAD", "ETNIA",
    "GLOSA_PAIS_ORIGEN", "PREVISION", "REGION_RESIDENCIA", "DIAG1_CATEGORIA",
]
ETIQUETA = "DIAS_ESTADA"
IDENTIFICADORES = ["ANO_FUENTE", "ID_REGISTRO"]
PARTICIONES = {"entrenamiento": 2023, "validacion": 2024, "prueba": 2025}

filas_base = pq.ParquetFile(ARCHIVO_COMUN).metadata.num_rows
with ARCHIVO_COMUN.open("rb") as archivo:
    sha256_base = hashlib.file_digest(archivo, "sha256").hexdigest()
versiones = {nombre: version(nombre) for nombre in ["pandas", "numpy", "scikit-learn", "scipy", "pyarrow", "joblib"]}
print(f"Base común: {filas_base:,} egresos. SHA-256: {sha256_base}")
display(pd.Series(versiones, name="version").to_frame())

Base común: 4,985,130 egresos. SHA-256: d14d4fd5ef489a4e9429eb9d9474bf400487596794bedbfdbfb7aae5e21bff0e


,version
pandas,3.1.1
numpy,2.5.4
scikit-learn,1.9.1
scipy,1.18.1
pyarrow,26.0.1
joblib,1.6.0


## 2. Separación temporal y codificación

- **2023: entrenamiento.** Único año utilizado para ajustar el codificador.
- **2024: validación.** Se transforma con el codificador de 2023.
- **2025: prueba final.** Se transforma de la misma forma y se reserva para la evaluación final del futuro modelo.

La asignación utiliza `ANO_FUENTE`, incluso si `ANO_EGRESO` está ausente. Los conteos de calidad de validación y prueba no se utilizan para ajustar la preparación.

Los ausentes de las ocho características se representan como `SIN_DATO`, sin imputar la respuesta más frecuente. Se conservan las categorías existentes, incluida región `NULL` (Extranjero) y `99` (Ignorada); no equivalen a un ausente.

**Categorías poco frecuentes:** se mantienen, con `min_frequency=None`. La tabla de entrenamiento contará las que tienen menos de 10 filas únicamente para describirlas: ese umbral no agrupa ni elimina datos. Los códigos diagnósticos sin correspondencia en el diccionario tampoco se borran ni se declaran inválidos.

**Categorías desconocidas:** `handle_unknown="ignore"` representa cada categoría no vista en 2023 mediante ceros en el bloque de esa característica. Se registran sus valores y cuántas filas afectan. No se equiparan con una categoría conocida ni se modifica el codificador al leer otro año.

Se usa **`drop=None`**: cada categoría aprendida tiene su indicador y ninguna categoría conocida queda representada por un bloque de ceros. Esto permite distinguir el caso desconocido y evita elegir ahora una referencia arbitraria. Con todas las indicadoras existen dependencias lineales; durante el modelado habrá que justificar restricciones, categorías de referencia o regularización antes de interpretar coeficientes individuales. [Documentación de OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).

In [3]:
codificador = OneHotEncoder(
    drop=None,
    min_frequency=None,
    handle_unknown="ignore",
    sparse_output=True,
    dtype=np.float32,
)

## 3. Preparación, verificaciones y guardado por año

Se lee un año y solo las 11 columnas necesarias: ocho características, la etiqueta y dos identificadores. Se conserva el orden de lectura y se crean `X`, `y` e identificadores desde las mismas filas, sin ordenarlos por separado.

No se eliminan repeticiones ni estadías extremas. Se comprueban etiquetas completas y positivas, identificadores únicos dentro de cada año, dimensiones y contenido de la matriz. El resumen documenta el máximo y los casos **superiores a 365 días**; ese umbral marca casos para revisión, no confirma un error. No se recorta, escala ni transforma `y`.

La codificación produce matrices CSR con valores de 32 bits, sin convertirlas en tablas densas. Cada archivo final `.joblib` reúne **`X`, `y`, `identificadores` y la huella de la base**. Se guarda una partición a la vez y luego se libera de memoria; no se crean copias categóricas exportadas ni matrices de desarrollo adicionales.

In [4]:
SALIDA.mkdir(parents=True, exist_ok=True)
resumen = []
control_categorias = []
perfil_entrenamiento = []

for nombre, anio in PARTICIONES.items():
    datos = pd.read_parquet(
        ARCHIVO_COMUN,
        columns=IDENTIFICADORES + CARACTERISTICAS + [ETIQUETA],
        filters=[("ANO_FUENTE", "==", anio)],
        dtype_backend="pyarrow",
    ).reset_index(drop=True)
    ausentes = datos[CARACTERISTICAS].isna().sum()
    X = datos[CARACTERISTICAS].fillna("SIN_DATO")
    y = datos[ETIQUETA]
    identificadores = datos[IDENTIFICADORES]
    del datos

    assert len(y) > 0 and y.notna().all() and y.gt(0).all(), "Revisar las etiquetas."
    assert identificadores["ANO_FUENTE"].eq(anio).all()
    assert identificadores["ID_REGISTRO"].is_unique
    assert X.index.equals(y.index) and y.index.equals(identificadores.index)

    if nombre == "entrenamiento":
        codificador.fit(X)

    bloques_desconocidos = 0
    for posicion, columna in enumerate(CARACTERISTICAS):
        nuevas = ~X[columna].isin(codificador.categories_[posicion])
        cantidad_nuevas = int(nuevas.sum())
        bloques_desconocidos += cantidad_nuevas
        control_categorias.append({
            "particion": nombre, "columna": columna,
            "ausentes_representados": int(ausentes[columna]),
            "filas_categoria_desconocida": cantidad_nuevas,
            "categorias_desconocidas": ", ".join(sorted(X.loc[nuevas, columna].unique().tolist())),
        })
        if nombre == "entrenamiento":
            frecuencias = X[columna].value_counts()
            poco_frecuentes = frecuencias[frecuencias < 10]
            perfil_entrenamiento.append({
                "columna": columna, "categorias_aprendidas": len(frecuencias),
                "categorias_menos_10_filas": len(poco_frecuentes),
                "filas_en_esas_categorias": int(poco_frecuentes.sum()),
            })

    X_numerica = codificador.transform(X)
    assert X_numerica.format == "csr"
    assert X_numerica.shape == (len(y), len(codificador.get_feature_names_out()))
    assert np.all(X_numerica.data == 1)
    assert X_numerica.nnz == len(y) * len(CARACTERISTICAS) - bloques_desconocidos

    archivo_salida = SALIDA / f"{nombre}_{anio}.joblib"
    joblib.dump({
        "X": X_numerica, "y": y, "identificadores": identificadores,
        "sha256_base": sha256_base,
    }, archivo_salida, compress=3)
    memoria = X_numerica.data.nbytes + X_numerica.indices.nbytes + X_numerica.indptr.nbytes
    resumen.append({
        "particion": nombre, "anio": anio, "filas_conservadas": len(y),
        "filas_eliminadas": 0, "columnas_X": X_numerica.shape[1],
        "etiquetas_ausentes": int(y.isna().sum()),
        "dias_minimos": int(y.min()), "mediana_dias": float(y.median()),
        "dias_maximos": int(y.max()), "estadias_mayores_365": int(y.gt(365).sum()),
        "memoria_X_MiB": round(memoria / 1024**2, 2),
        "archivo_MiB": round(archivo_salida.stat().st_size / 1024**2, 2),
    })
    print(f"{nombre} ({anio}): {len(y):,} filas, {X_numerica.shape[1]:,} columnas. Guardado: {archivo_salida.name}")
    del X, X_numerica, y, identificadores, nuevas

entrenamiento (2023): 1,612,267 filas, 1,516 columnas. Guardado: entrenamiento_2023.joblib
validacion (2024): 1,667,349 filas, 1,516 columnas. Guardado: validacion_2024.joblib
prueba (2025): 1,705,514 filas, 1,516 columnas. Guardado: prueba_2025.joblib


## 4. Resultado y control de categorías

Las tablas siguientes muestran conservación de filas, extremos, ausentes representados y categorías nuevas. Las categorías poco frecuentes de entrenamiento se mantienen en la matriz. Una fila puede tener más de una característica desconocida: los conteos por columna no deben sumarse como si fueran personas distintas.

In [5]:
resumen = pd.DataFrame(resumen)
control_categorias = pd.DataFrame(control_categorias)
perfil_entrenamiento = pd.DataFrame(perfil_entrenamiento)
assert int(resumen["filas_conservadas"].sum()) == filas_base
assert resumen["columnas_X"].nunique() == 1

print("Resumen de las tres particiones:")
display(resumen)
print("Categorías aprendidas en 2023; menos de 10 filas es solo un criterio descriptivo:")
display(perfil_entrenamiento)
print("Ausentes y categorías desconocidas por característica y partición:")
display(control_categorias)
print(f"Máximo observado: {resumen['dias_maximos'].max():,} días. Estadías >365 conservadas: {resumen['estadias_mayores_365'].sum():,}.")

Resumen de las tres particiones:


,particion,anio,filas_conservadas,filas_eliminadas,columnas_X,etiquetas_ausentes,dias_minimos,mediana_dias,dias_maximos,estadias_mayores_365,memoria_X_MiB,archivo_MiB
0,entrenamiento,2023,1612267,0,1516,0,1,2.0,22239,583,104.56,9.49
1,validacion,2024,1667349,0,1516,0,1,2.0,17858,593,108.13,14.69
2,prueba,2025,1705514,0,1516,0,1,2.0,20368,637,110.60,15.03


Categorías aprendidas en 2023; menos de 10 filas es solo un criterio descriptivo:


,columna,categorias_aprendidas,categorias_menos_10_filas,filas_en_esas_categorias
0,PERTENENCIA_ESTABLECIMIENTO_SALU,3,0,0
1,SEXO,3,0,0
2,GRUPO_EDAD,12,0,0
3,ETNIA,3,0,0
4,GLOSA_PAIS_ORIGEN,3,0,0
5,PREVISION,8,0,0
6,REGION_RESIDENCIA,19,0,0
7,DIAG1_CATEGORIA,1465,170,687


Ausentes y categorías desconocidas por característica y partición:


,particion,columna,ausentes_representados,filas_categoria_desconocida,categorias_desconocidas
0,entrenamiento,PERTENENCIA_ESTABLECIMIENTO_SALU,128108,0,
1,entrenamiento,SEXO,128108,0,
2,entrenamiento,GRUPO_EDAD,128108,0,
3,entrenamiento,ETNIA,128108,0,
4,entrenamiento,GLOSA_PAIS_ORIGEN,128108,0,
5,entrenamiento,PREVISION,128108,0,
6,entrenamiento,REGION_RESIDENCIA,128108,0,
7,entrenamiento,DIAG1_CATEGORIA,0,0,
8,validacion,PERTENENCIA_ESTABLECIMIENTO_SALU,124508,0,
9,validacion,SEXO,124508,0,


Máximo observado: 22,239 días. Estadías >365 conservadas: 1,813.


## 5. Reutilización en la fase de modelado

Se guarda el codificador de 2023 junto con el orden de características, las versiones, la huella de la fuente y los controles agregados. Los nombres y el orden de las columnas numéricas se recuperan con `get_feature_names_out()`; no hace falta otro CSV.

In [6]:
joblib.dump({
    "codificador": codificador, "caracteristicas": CARACTERISTICAS,
    "etiqueta": ETIQUETA, "identificadores": IDENTIFICADORES,
    "particiones": PARTICIONES, "sha256_base": sha256_base, "versiones": versiones,
    "resumen": resumen, "perfil_entrenamiento": perfil_entrenamiento,
    "control_categorias": control_categorias,
}, SALIDA / "preprocesador_2023.joblib", compress=3)
print("Preparación terminada. Cuatro archivos finales en:", SALIDA)

Preparación terminada. Cuatro archivos finales en: datos\procesados\regresion


Ejemplo de lectura para la siguiente fase, **sin ejecutarlo aquí ni cargar matrices duplicadas**:

```python
particion = joblib.load("datos/procesados/regresion/entrenamiento_2023.joblib")
X = particion["X"]
y = particion["y"]
identificadores = particion["identificadores"]
preparacion = joblib.load("datos/procesados/regresion/preprocesador_2023.joblib")
nombres_columnas = preparacion["codificador"].get_feature_names_out()
assert particion["sha256_base"] == preparacion["sha256_base"]
```

`X[i]`, `y.iloc[i]` e `identificadores.iloc[i]` corresponden al mismo egreso. Al seleccionar o reordenar filas, aplicar los mismos índices a los tres objetos. Los identificadores no son entradas del algoritmo.

La codificación actual contiene todas las categorías aprendidas y bloques de ceros para desconocidas; esa distinción debe conservarse al interpretar resultados. No se eliminan categorías raras ni se presume que un código ausente del diccionario sea inválido.

Para seleccionar y ajustar modelos se utilizarán entrenamiento y validación. Una vez decidida la metodología, se podrá volver a ajustar el preprocesamiento con **2023–2024 como desarrollo completo** y transformar nuevamente 2025 con ese nuevo codificador. No se deben concatenar matrices generadas con codificadores diferentes ni usar 2025 para elegir transformaciones o parámetros. Si se utiliza validación cruzada, ajustar la preparación dentro de cada partición de entrenamiento. [Prevención de fuga de información](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).

El tratamiento definitivo de estadías extremas, la interpretación de coeficientes y la evaluación del rendimiento quedan para el modelado. Este notebook no los resuelve ni modifica la base común.

In [7]:
display(X.head(10))
display(y.head(10))

NameError: name 'X' is not defined